# Comparing Simple and Multiple Linear Regression

In this notebook, we will create a reproducible synthetic dataset and use scikit-learn to compare two ways of predicting a continuous target. We will train both models on the same training data, evaluate them on held-out test data, and visualize example prediction errors.

## 1. Import the required libraries

The notebook uses NumPy to generate data, pandas to work with the CSV, Matplotlib for plots, and scikit-learn for splitting, training, and evaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2. Generate and save the synthetic dataset

We create 5,000 observations with four numerical input features. The target is a linear combination of those features plus random noise, so there is a learnable relationship but predictions will not be perfectly exact. A fixed random seed makes the generated data reproducible.

In [ ]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
n_observations = 5_000

experience_years = rng.uniform(0, 30, size=n_observations)
education_years = rng.uniform(8, 22, size=n_observations)
projects_completed = rng.integers(0, 21, size=n_observations)
weekly_study_hours = rng.uniform(0, 20, size=n_observations)
noise = rng.normal(0, 8, size=n_observations)

target_score = (
    50
    + 2.5 * experience_years
    + 3.0 * education_years
    + 1.2 * projects_completed
    + 0.8 * weekly_study_hours
    + noise
)

generated_data = pd.DataFrame(
    {
        "experience_years": experience_years,
        "education_years": education_years,
        "projects_completed": projects_completed,
        "weekly_study_hours": weekly_study_hours,
        "target_score": target_score,
    }
)

csv_path = Path.cwd() / "regression_data.csv"
generated_data.to_csv(csv_path, index=False)
print(f"Saved {len(generated_data):,} observations to {csv_path.resolve()}")

## 3. Load and inspect the data

We load the CSV we just created into a pandas DataFrame. When you run this notebook with its folder as the working directory (the usual Jupyter behavior), `regression_data.csv` is saved beside the notebook.

In [ ]:
data = pd.read_csv(csv_path)

print(f"Dataset shape (rows, columns): {data.shape}")
display(data.head())
display(data.describe().round(2))

## 4. Choose inputs and split into training and test sets

**Simple Linear Regression (SLR)** uses one input feature to predict a target. Here it uses `experience_years`. **Multiple Linear Regression (MLR)** uses more than one input feature; here it uses all four features. Both are linear regression models, but MLR can account for the contributions of several inputs at once.

We reserve 20% of the observations as test data. The models will learn only from the training portion; the test portion is held out for an unbiased evaluation.

In [ ]:
feature_columns = [
    "experience_years",
    "education_years",
    "projects_completed",
    "weekly_study_hours",
]
target_column = "target_score"

X = data[feature_columns]
y = data[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

X_train_slr = X_train[["experience_years"]]
X_test_slr = X_test[["experience_years"]]
X_train_mlr = X_train[feature_columns]
X_test_mlr = X_test[feature_columns]

print(f"Training observations: {len(X_train):,}")
print(f"Testing observations:  {len(X_test):,}")

## 5. Train both models and make test-set predictions

Each model is fit using only its training features and the training targets. We then use each fitted model to predict targets for the same held-out test observations.

In [ ]:
slr_model = LinearRegression()
slr_model.fit(X_train_slr, y_train)
slr_predictions = slr_model.predict(X_test_slr)

mlr_model = LinearRegression()
mlr_model.fit(X_train_mlr, y_train)
mlr_predictions = mlr_model.predict(X_test_mlr)

print(f"SLR feature: {X_train_slr.columns[0]}")
print(f"SLR coefficient: {slr_model.coef_[0]:.3f}")
print(f"MLR features: {', '.join(X_train_mlr.columns)}")
print(f"MLR coefficients: {dict(zip(X_train_mlr.columns, mlr_model.coef_))}")

## 6. Evaluate model performance

We calculate all three metrics on the held-out test set:

- **Mean Squared Error (MSE):** the average squared difference between actual and predicted values. Lower is better; squaring makes large errors count more heavily, and the units are squared.
- **Mean Absolute Error (MAE):** the average absolute difference between actual and predicted values. Lower is better, and the result is in the target's original units.
- **R² score:** the proportion of target variation explained by the model relative to predicting the test-set mean. A score closer to 1 indicates a better fit; 0 is comparable to that mean baseline, and a negative score is worse than the baseline.

In [ ]:
results = pd.DataFrame(
    {
        "Model": ["Simple Linear Regression (SLR)", "Multiple Linear Regression (MLR)"],
        "MSE": [
            mean_squared_error(y_test, slr_predictions),
            mean_squared_error(y_test, mlr_predictions),
        ],
        "MAE": [
            mean_absolute_error(y_test, slr_predictions),
            mean_absolute_error(y_test, mlr_predictions),
        ],
        "R²": [
            r2_score(y_test, slr_predictions),
            r2_score(y_test, mlr_predictions),
        ],
    }
)
display(results.set_index("Model").round(3))

## 7. Plot 10 test observations and their residuals

A **residual** is `actual - predicted`. In each plot, the vertical connecting segment between the actual and predicted marker represents that observation's residual. The table below each plot gives the actual value, prediction, and exact residual for all 10 plotted test observations.

In [ ]:
n_plot = 10
actual_values = y_test.iloc[:n_plot].to_numpy()
observation_numbers = np.arange(1, n_plot + 1)

def plot_predictions(title, predictions):
    plotted_predictions = predictions[:n_plot]
    residuals = actual_values - plotted_predictions

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.vlines(
        observation_numbers,
        plotted_predictions,
        actual_values,
        color="gray",
        alpha=0.65,
        linewidth=2,
        label="Residual (actual - predicted)",
    )
    ax.scatter(observation_numbers, actual_values, color="tab:blue", s=55, label="Actual")
    ax.scatter(
        observation_numbers,
        plotted_predictions,
        color="tab:orange",
        marker="x",
        s=65,
        label="Predicted",
    )
    ax.set_title(title)
    ax.set_xlabel("Test observation number (first 10 observations)")
    ax.set_ylabel("Target score")
    ax.set_xticks(observation_numbers)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()

    residual_table = pd.DataFrame(
        {
            "Observation": observation_numbers,
            "Actual": actual_values,
            "Predicted": plotted_predictions,
            "Residual (actual - predicted)": residuals,
        }
    )
    display(residual_table.round(2))

plot_predictions("SLR: Actual vs Predicted for 10 Test Observations", slr_predictions)
plot_predictions("MLR: Actual vs Predicted for 10 Test Observations", mlr_predictions)

## 8. Interpreting the plots and residuals

A **positive residual** means the actual value is greater than the prediction, so the model underpredicted. A **negative residual** means the prediction was greater than the actual value, so the model overpredicted. The absolute size of a residual indicates how far the prediction was from that observation's actual value; shorter connecting segments indicate smaller errors.

These plots show only 10 test examples, so they help us inspect individual predictions but do not describe performance across the whole test set. Use MSE, MAE, and R² above to compare overall performance. MLR can use the other features' signal that SLR does not see, though the exact benefit depends on the relationship in the data.